In [ ]:
# ============================================================================
# COMPLETE TESTING CODE - ALL 6 MODELS WITH FAIR DATASET
# Dataset has BOTH answers with SAME case marker built-in
# ============================================================================

import pandas as pd
import numpy as np
import torch
import json
from transformers import AutoTokenizer, AutoModelForMaskedLM

# ============================================================================
# STEP 1: Load FAIR dataset
# ============================================================================

print("📂 Loading FAIR dataset (both answers have same case)...")
with open('uzbek_psych_verbs_FINAL.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)
print(f"✅ Loaded {len(dataset)} items")
print(f"   All items have both answers with SAME case marker!\n")


# ============================================================================
# STEP 2: Define testing function (simplified)
# ============================================================================

def calculate_perplexity(model, tokenizer, text):
    """Calculate perplexity for a text sequence."""
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs['input_ids'])
        loss = outputs.loss.item()
    return np.exp(loss)


def test_item(item, model, tokenizer):
    """
    Test single item - answers already have same case in dataset.
    """
    # Build context
    context = f"{item['verb_definition']}\n\n{item['story_part1']}\n\n{item['story_part2']}\n\n"
    question = item['test_question']

    # Get answers (already have same case in dataset!)
    answer1 = item['answer_choices']['answer_1']
    answer2 = item['answer_choices']['answer_2']

    # Build full prompts
    text1 = f"{context}{question} {answer1}"
    text2 = f"{context}{question} {answer2}"

    # Calculate perplexity
    ppl1 = calculate_perplexity(model, tokenizer, text1)
    ppl2 = calculate_perplexity(model, tokenizer, text2)

    # EO preference = 1 if model prefers answer_2 (the EO-structure animal)
    eo_pref = 1 if ppl2 < ppl1 else 0

    return {
        'item_id': item['id'],
        'condition': item['condition'],
        'verb': item['verb'],
        'ppl_answer1': ppl1,
        'ppl_answer2': ppl2,
        'eo_preference': eo_pref
    }

print("✅ Testing function defined!\n")


# ============================================================================
# STEP 3: Test mBERT
# ============================================================================

print("🔄 Loading mBERT (180M params)...")
tokenizer_mbert = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')
model_mbert = AutoModelForMaskedLM.from_pretrained('bert-base-multilingual-cased')
model_mbert.eval()
print("✅ mBERT loaded!\n")

print("🧪 Testing 32 items with mBERT...\n")
results_mbert = []
for i, item in enumerate(dataset, 1):
    print(f"  {i}/32: {item['id'][:20]}...", end=' ')
    result = test_item(item, model_mbert, tokenizer_mbert)
    results_mbert.append(result)
    print(f"✓ (EO={result['eo_preference']})")

df_mbert = pd.DataFrame(results_mbert)
print("\n✅ mBERT complete!\n")


# ============================================================================
# STEP 4: Test XLM-RoBERTa-base
# ============================================================================

print("🔄 Loading XLM-RoBERTa-base (270M params)...")
tokenizer_base = AutoTokenizer.from_pretrained('xlm-roberta-base')
model_base = AutoModelForMaskedLM.from_pretrained('xlm-roberta-base')
model_base.eval()
print("✅ XLM-RoBERTa-base loaded!\n")

print("🧪 Testing 32 items with XLM-RoBERTa-base...\n")
results_base = []
for i, item in enumerate(dataset, 1):
    print(f"  {i}/32: {item['id'][:20]}...", end=' ')
    result = test_item(item, model_base, tokenizer_base)
    results_base.append(result)
    print(f"✓ (EO={result['eo_preference']})")

df_base = pd.DataFrame(results_base)
print("\n✅ XLM-RoBERTa-base complete!\n")


# ============================================================================
# STEP 5: Test BERTbek
# ============================================================================

print("🔄 Loading BERTbek (Uzbek-specific)...")
tokenizer_bertbek = AutoTokenizer.from_pretrained('elmurod1202/bertbek-news-big-cased')
model_bertbek = AutoModelForMaskedLM.from_pretrained('elmurod1202/bertbek-news-big-cased')
model_bertbek.eval()
print("✅ BERTbek loaded!\n")

print("🧪 Testing 32 items with BERTbek...\n")
results_bertbek = []
for i, item in enumerate(dataset, 1):
    print(f"  {i}/32: {item['id'][:20]}...", end=' ')
    result = test_item(item, model_bertbek, tokenizer_bertbek)
    results_bertbek.append(result)
    print(f"✓ (EO={result['eo_preference']})")

df_bertbek = pd.DataFrame(results_bertbek)
print("\n✅ BERTbek complete!\n")


# ============================================================================
# STEP 6: Test XLM-RoBERTa-large
# ============================================================================

print("🔄 Loading XLM-RoBERTa-large (550M params)...")
tokenizer_large = AutoTokenizer.from_pretrained('xlm-roberta-large')
model_large = AutoModelForMaskedLM.from_pretrained('xlm-roberta-large')
model_large.eval()
print("✅ XLM-RoBERTa-large loaded!\n")

print("🧪 Testing 32 items with XLM-RoBERTa-large...\n")
results_large = []
for i, item in enumerate(dataset, 1):
    print(f"  {i}/32: {item['id'][:20]}...", end=' ')
    result = test_item(item, model_large, tokenizer_large)
    results_large.append(result)
    print(f"✓ (EO={result['eo_preference']})")

df_large = pd.DataFrame(results_large)
print("\n✅ XLM-RoBERTa-large complete!\n")


# ============================================================================
# STEP 7: Test XLM-V-base
# ============================================================================

print("🔄 Loading XLM-V-base (1.2B params)...")
print("   (This is large - download may take 5-10 min)")
tokenizer_xlmv = AutoTokenizer.from_pretrained('facebook/xlm-v-base')
model_xlmv = AutoModelForMaskedLM.from_pretrained('facebook/xlm-v-base')
model_xlmv.eval()
print("✅ XLM-V loaded!\n")

print("🧪 Testing 32 items with XLM-V...\n")
results_xlmv = []
for i, item in enumerate(dataset, 1):
    print(f"  {i}/32: {item['id'][:20]}...", end=' ')
    result = test_item(item, model_xlmv, tokenizer_xlmv)
    results_xlmv.append(result)
    print(f"✓ (EO={result['eo_preference']})")

df_xlmv = pd.DataFrame(results_xlmv)
print("\n✅ XLM-V complete!\n")


# ============================================================================
# STEP 8: Calculate and display results
# ============================================================================

print("\n" + "="*70)
print("📊 FINAL RESULTS - FAIR METHOD (Same Case Markers)")
print("="*70)

# Calculate means
mbert_es = df_mbert[df_mbert['condition']=='ES']['eo_preference'].mean() * 100
mbert_eo = df_mbert[df_mbert['condition']=='EO']['eo_preference'].mean() * 100
mbert_diff = mbert_eo - mbert_es

base_es = df_base[df_base['condition']=='ES']['eo_preference'].mean() * 100
base_eo = df_base[df_base['condition']=='EO']['eo_preference'].mean() * 100
base_diff = base_eo - base_es

bertbek_es = df_bertbek[df_bertbek['condition']=='ES']['eo_preference'].mean() * 100
bertbek_eo = df_bertbek[df_bertbek['condition']=='EO']['eo_preference'].mean() * 100
bertbek_diff = bertbek_eo - bertbek_es

large_es = df_large[df_large['condition']=='ES']['eo_preference'].mean() * 100
large_eo = df_large[df_large['condition']=='EO']['eo_preference'].mean() * 100
large_diff = large_eo - large_es

xlmv_es = df_xlmv[df_xlmv['condition']=='ES']['eo_preference'].mean() * 100
xlmv_eo = df_xlmv[df_xlmv['condition']=='EO']['eo_preference'].mean() * 100
xlmv_diff = xlmv_eo - xlmv_es

child_es = 36.0
child_eo = 76.0
child_diff = 40.0

# Display table
print("\nEO Preference Rate by Condition:\n")
print(f"{'Model':<30} {'Year':<6} {'Size':<8} {'ES Cond':<10} {'EO Cond':<10} {'Difference'}")
print("-" * 70)
print(f"{'Children (N=40)':<30} {'-':<6} {'N/A':<8} {child_es:>6.1f}%   {child_eo:>6.1f}%   {child_diff:>6.1f} pp")
print(f"{'mBERT':<30} {'2018':<6} {'180M':<8} {mbert_es:>6.1f}%   {mbert_eo:>6.1f}%   {mbert_diff:>6.1f} pp")
print(f"{'XLM-RoBERTa-base':<30} {'2019':<6} {'270M':<8} {base_es:>6.1f}%   {base_eo:>6.1f}%   {base_diff:>6.1f} pp")
print(f"{'BERTbek (Uzbek-specific)':<30} {'2024':<6} {'~110M':<8} {bertbek_es:>6.1f}%   {bertbek_eo:>6.1f}%   {bertbek_diff:>6.1f} pp")
print(f"{'XLM-RoBERTa-large':<30} {'2019':<6} {'550M':<8} {large_es:>6.1f}%   {large_eo:>6.1f}%   {large_diff:>6.1f} pp")
print(f"{'XLM-V-base':<30} {'2023':<6} {'1.2B':<8} {xlmv_es:>6.1f}%   {xlmv_eo:>6.1f}%   {xlmv_diff:>6.1f} pp")

print("\n" + "="*70)
print("\n✅ FAIR METHOD: Both answers had SAME case marker.")
print("   Models could NOT use morphological cues.")
print("   This matches how children were tested!")


# ============================================================================
# STEP 9: Save results
# ============================================================================

df_mbert['model'] = 'mBERT'
df_base['model'] = 'XLM-RoBERTa-base'
df_bertbek['model'] = 'BERTbek'
df_large['model'] = 'XLM-RoBERTa-large'
df_xlmv['model'] = 'XLM-V-base'

df_all = pd.concat([df_mbert, df_base, df_bertbek, df_large, df_xlmv])
df_all.to_csv('all_models_FAIR_results.csv', index=False)
print("\n💾 Results saved to 'all_models_FAIR_results.csv'")

summary = pd.DataFrame({
    'Model': ['Children', 'mBERT', 'XLM-RoBERTa-base', 'BERTbek', 'XLM-RoBERTa-large', 'XLM-V-base'],
    'Year': ['-', '2018', '2019', '2024', '2019', '2023'],
    'Parameters': ['N/A', '180M', '270M', '~110M', '550M', '1.2B'],
    'ES_Preference_%': [child_es, mbert_es, base_es, bertbek_es, large_es, xlmv_es],
    'EO_Preference_%': [child_eo, mbert_eo, base_eo, bertbek_eo, large_eo, xlmv_eo],
    'Difference_pp': [child_diff, mbert_diff, base_diff, bertbek_diff, large_diff, xlmv_diff]
})

summary.to_csv('summary_FAIR_results.csv', index=False)
print("💾 Summary saved to 'summary_FAIR_results.csv'")

print("\n✅ ALL DONE!")

In [ ]:
# ============================================================================
# PLOTTING & SAVE
# ============================================================================

import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd

# ── Collect results ──────────────────────────────────────────────────────────
child_es, child_eo = 36.0, 76.0

rows = [
    ('Children (N=40)',    '-',    'N/A',   child_es,    child_eo),
    ('mBERT',             '2018', '180M',  mbert_es,    mbert_eo),
    ('XLM-RoBERTa-base',  '2019', '270M',  base_es,     base_eo),
    ('BERTbek',           '2024', '~110M', bertbek_es,  bertbek_eo),
    ('XLM-RoBERTa-large', '2019', '550M',  large_es,    large_eo),
    ('XLM-V-base',        '2023', '1.2B',  xlmv_es,     xlmv_eo),
]

# ── Bar chart ────────────────────────────────────────────────────────────────
model_labels = [
    'Children\n(N=40)', 'mBERT\n(180M,2018)', 'XLM-R-base\n(270M,2019)',
    'BERTbek\n(Uzbek,2024)', 'XLM-R-large\n(550M,2019)', 'XLM-V-base\n(1.2B,2023)'
]
es_vals = [r[3] for r in rows]
eo_vals = [r[4] for r in rows]

x = np.arange(len(model_labels))
w = 0.35
fig, ax = plt.subplots(figsize=(13, 6.5))
ax.bar(x - w/2, es_vals, w, color='#E8835A', label='ES (habitual)', zorder=3)
ax.bar(x + w/2, eo_vals, w, color='#2A7F7F', label='EO (episodic)', zorder=3)
ax.axhline(50, color='gray', linestyle='--', linewidth=1.2, alpha=0.6, label='Chance (50%)')
ax.set_xticks(x)
ax.set_xticklabels(model_labels, fontsize=10)
ax.set_ylabel('EO Preference (%)', fontsize=12)
ax.set_ylim(0, 100)
ax.set_title(
    'Argument Linking Preferences: Children vs. LLMs (No Overt Case Marker)\n'
    'Both Novel Verbs: navamoq + payramoq | Same Case Markers',
    fontsize=13, fontweight='bold'
)
ax.legend(title='Condition', frameon=True)
plt.tight_layout()
plt.savefig('results_no_overt_case.png', dpi=200, bbox_inches='tight')
plt.show()
print('📊 Bar chart saved as results_no_overt_case.png')

# ── Difference chart ─────────────────────────────────────────────────────────
diffs = [r[4] - r[3] for r in rows]
colors = ['#2CA02C' if d > 15 else '#F0A830' if d > 0 else '#E84040' for d in diffs]
colors[0] = '#2CA02C'  # children always green

fig2, ax2 = plt.subplots(figsize=(11, 5.5))
bars = ax2.bar(model_labels, diffs, color=colors, width=0.5, zorder=3,
               edgecolor='#333333', linewidth=1)
for bar, val in zip(bars, diffs):
    ypos = val + 0.8 if val >= 0 else val - 2.5
    ax2.text(bar.get_x() + bar.get_width()/2, ypos,
             f'{val:.1f}pp', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.axhline(0, color='#888888', linewidth=0.8, alpha=0.5)
ax2.axhline(child_eo - child_es, color='#2CA02C', linestyle='--',
            linewidth=2, label=f'Children ({child_eo-child_es:.0f}pp)')
ax2.set_ylabel('Difference (EO − ES) in percentage points', fontsize=12)
ax2.set_title('Pattern Strength: How Much Do Models Distinguish ES vs EO?\n(No Overt Case Marker)',
              fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
plt.tight_layout()
plt.savefig('results_no_overt_case_differences.png', dpi=200, bbox_inches='tight')
plt.show()
print('📊 Difference chart saved as results_no_overt_case_differences.png')

# ── Save CSVs ────────────────────────────────────────────────────────────────
df_mbert['model']   = 'mBERT'
df_base['model']    = 'XLM-RoBERTa-base'
df_bertbek['model'] = 'BERTbek'
df_large['model']   = 'XLM-RoBERTa-large'
df_xlmv['model']    = 'XLM-V-base'

df_all = pd.concat([df_mbert, df_base, df_bertbek, df_large, df_xlmv])
df_all['method'] = 'no_overt_case_marker'
df_all.to_csv('results_no_overt_case_detailed.csv', index=False)

summary = pd.DataFrame({
    'Model':         [r[0] for r in rows],
    'Year':          [r[1] for r in rows],
    'Parameters':    [r[2] for r in rows],
    'ES_Pref_%':     [r[3] for r in rows],
    'EO_Pref_%':     [r[4] for r in rows],
    'Difference_pp': [r[4]-r[3] for r in rows],
    'Method':        ['no_overt_case_marker'] * 6
})
summary.to_csv('results_no_overt_case_summary.csv', index=False)

from google.colab import files
files.download('results_no_overt_case_summary.csv')
files.download('results_no_overt_case_detailed.csv')
files.download('results_no_overt_case.png')
files.download('results_no_overt_case_differences.png')
print('\n✅ ALL DONE! All files downloaded.')